# WV3-Transfer HAT Quick Inference
Upload LR-MS + PAN + `best_wv3_transfer_hat_6band.pth`.

In [ ]:
!pip -q install rasterio affine timm tqdm
!git clone -q https://github.com/abobakerkamel/satellite-pansharpening-research.git
%cd satellite-pansharpening-research
!python models/hat/prepare_hat_arch.py

In [ ]:
from google.colab import files
uploaded=files.upload()
print('Upload LR-MS .tif, PAN .tif and HAT .pth')

In [ ]:
from pathlib import Path
import torch
from models.hat.model import HATPanFusion
from models.common.inference_utils import full_scene_tiled_inference
names=list(uploaded);ck=Path(next(n for n in names if n.endswith('.pth')));tifs=[n for n in names if n.lower().endswith(('.tif','.tiff'))]
MS_PATH=Path(tifs[0]);PAN_PATH=Path(tifs[1]);OUT=Path('HAT_HRMS_6band.tif')
d=torch.device('cuda' if torch.cuda.is_available() else 'cpu');m=HATPanFusion().to(d);c=torch.load(ck,map_location=d,weights_only=False);m.load_state_dict(c['model_state_dict'],strict=True)
full_scene_tiled_inference(m,MS_PATH,PAN_PATH,'weights/train_normalization_stats.json',OUT,tile_lr=128,device=d);print(OUT)

In [ ]:
from google.colab import files
files.download('HAT_HRMS_6band.tif')